# Setup del entorno en Google Colab — Tesis KDM vs NPC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Steven10P/KDM_NPC/blob/main/notebooks/00_setup_colab.ipynb)

**Análisis Comparativo del Desempeño entre Kernel Density Matrix y Neural Probabilistic Circuits**

> ⚠️ **Ruta GPU primaria: Kaggle Kernels** (push/status/output vía CLI, sin
> pasos manuales de navegador). Este notebook de Colab queda como **respaldo
> manual** para cuando se agote la cuota gratuita de Kaggle (~30h GPU/semana)
> o para depuración interactiva.

Este notebook prepara el entorno de trabajo en Colab:
1. Monta Google Drive (para persistir datasets, checkpoints y resultados).
2. Clona el repositorio de la tesis y los repositorios de referencia (incluye `learnspn`, necesario para NPC(Data)).
3. Instala `kdm-torch` (librería KDM) y `mlflow` (registro de experimentos, tracker principal del proyecto).
4. Verifica la instalación con un ejemplo mínimo de clasificación con KDM.

> ⚡ En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)` antes de entrenar modelos.

In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Carpeta de trabajo persistente en Drive
import os
WORKDIR = '/content/drive/MyDrive/Tesis_KDM_NPC'
os.makedirs(WORKDIR, exist_ok=True)
print('Carpeta de trabajo:', WORKDIR)

In [ ]:
# 2. Clonar repositorios (tesis + referencias)
%cd /content
!git clone https://github.com/Steven10P/KDM_NPC.git
!git clone --depth 1 https://github.com/fagonzalezo/kdm.git
!git clone --depth 1 https://github.com/uiuctml/npc-models.git
!git clone --depth 1 https://github.com/uiuctml/npc-dataset-utils.git
!git clone --depth 1 https://github.com/uiuctml/learnspn.git
# learnspn requiere Java (openjdk-17) + bash; solo corre en Linux — Colab/Kaggle cumplen, Windows local no
!apt-get -qq install -y openjdk-17-jdk > /dev/null

In [ ]:
# 3. Instalar librerías
!pip install -q git+https://github.com/fagonzalezo/kdm.git mlflow
# Dependencias de npc-models (instalar cuando se vaya a entrenar NPC;
# ojo: npc-models fija torch==2.1.2 y kdm-torch requiere torch>=2.2 —
# usar entornos/sesiones separadas para cada modelo si hay conflicto)
# !pip install -q -r /content/npc-models/requirements.txt

In [ ]:
# 4. Verificación: clasificación mínima con KDM (dos lunas)
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from kdm.models import KDMClassModel
from kdm.init import init_kdm_layer

X, y = make_moons(n_samples=1000, noise=0.15, random_state=42)
X = torch.tensor(X, dtype=torch.float32); y = torch.tensor(y)
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

encoder = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 8))
model = KDMClassModel(encoded_size=8, dim_y=2, encoder=encoder, n_comp=64, sigma=0.5)

x_init, y_init = X[:64], torch.nn.functional.one_hot(y[:64], 2).float()
init_kdm_layer(model.kdm, encoder(x_init).detach(), y_init, init_sigma=True)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    for xb, yb in loader:
        probs = model(xb)
        loss = nn.functional.nll_loss(torch.log(probs + 1e-8), yb)
        opt.zero_grad(); loss.backward(); opt.step()

acc = (model(X).argmax(1) == y).float().mean().item()
print(f'KDM funcionando — exactitud en dos lunas: {acc:.3f}')

In [ ]:
# 5. Conectar MLflow para registrar experimentos (tracker principal del proyecto)
# El backend sqlite (mlflow.db) vive en Drive para persistir entre sesiones:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{WORKDIR}/mlflow.db")
mlflow.set_experiment("tesis-kdm-npc")

## Próximos pasos

- **Estructura de experimentos**: `experiments/exp_N/` (`DESIGN.md` → `IMPLEMENTATION.md` → `scripts/configs/` → `results/<condición>/` → `reports/`). Ver `experiments/INDEX.md` en el repo.
- **Datasets NPC** (MNIST-Addition, GTSRB, CelebA, AwA2): usar las utilidades de `/content/npc-dataset-utils` y guardarlos en `WORKDIR` para no re-descargarlos en cada sesión.
- **LearnSPN** (`/content/learnspn`, requiere el JDK ya instalado arriba): genera la estructura del circuito para NPC(Data) — ver `external/learnspn/README.md` en el repo para el formato de datos esperado.
- **Ejemplos KDM**: `/content/kdm/examples/` (`kdm_classification`, `kdm_regression`, `kdm_density_estimation` ya portados a PyTorch).
- **Modelos NPC**: `/content/npc-models` — entrenamiento en 3 etapas (atributos → circuito → optimización conjunta).
- **Ruta GPU primaria**: Kaggle Kernels (push/status/output vía CLI). Este notebook es el respaldo manual.
- **Papers**: [NPC — Chen et al. 2025](https://arxiv.org/abs/2501.07021) · [KDM — González et al. 2025](https://doi.org/10.1007/s42484-025-00299-9)